# Chapter 8 - Data Modification
## Individual Assignment  -  10 Propositions & Queries
**Name:** Kanwal Jit Singh  
**Group:** 2  
**Database:** Northwinds2024Student  
**Reference:** T-SQL Fundamentals, Chapter 8  

All my queries use staging tables prefixed with `KS_` under the `dbo` schema so I dont touch the original Sales/Production tables directly.

In [ ]:
USE Northwinds2024Student;
GO

---
## Proposition 1 - SELECT INTO
**Problem:** I need a working copy of orders from 2020 through 2022 so I can practice data modification without messing up the real Sales.Orders table. Use SELECT INTO to create dbo.KS_Orders with just those rows.

In [ ]:
DROP TABLE IF EXISTS dbo.KS_Orders;
GO
SELECT *
INTO dbo.KS_Orders
FROM Sales.Orders
WHERE orderdate >= '20200101'
  AND orderdate < '20230101';
GO
SELECT COUNT(*) AS total_rows FROM dbo.KS_Orders;

---
## Proposition 2 - INSERT VALUES
**Problem:** I want to create a customers staging table and add a brand new customer row (custid 100, Coho Winery from Redmond WA) using INSERT VALUES with an explicit column list.

In [ ]:
DROP TABLE IF EXISTS dbo.KS_Customers;
GO
CREATE TABLE dbo.KS_Customers
(
  custid      INT          NOT NULL PRIMARY KEY,
  companyname NVARCHAR(40) NOT NULL,
  country     NVARCHAR(15) NOT NULL,
  region      NVARCHAR(15) NULL,
  city        NVARCHAR(15) NOT NULL
);
GO
INSERT INTO dbo.KS_Customers(custid, companyname, country, region, city)
VALUES(100, N'Coho Winery', N'USA', N'WA', N'Redmond');
SELECT * FROM dbo.KS_Customers WHERE custid = 100;

---
## Proposition 3 - INSERT SELECT with EXISTS
**Problem:** Now I want to fill dbo.KS_Customers with all customers from Sales.Customers who have actually placed at least one order. I will use INSERT SELECT combined with an EXISTS subquery to filter them.

In [ ]:
INSERT INTO dbo.KS_Customers(custid, companyname, country, region, city)
SELECT C.custid, C.companyname, C.country, C.region, C.city
FROM Sales.Customers AS C
WHERE EXISTS
  (SELECT * FROM Sales.Orders AS O
   WHERE O.custid = C.custid)
AND C.custid <> 100;
SELECT country, COUNT(*) AS cnt
FROM dbo.KS_Customers
GROUP BY country
ORDER BY cnt DESC;

---
## Proposition 4 - UPDATE with OUTPUT (fix NULL regions)
**Problem:** Some customers in my staging table have NULL in the region column. I want to update those to the string '<None>' and use the OUTPUT clause to see exactly which rows changed, showing custid, the old region, and the new region.

In [ ]:
SELECT COUNT(*) AS null_regions_before
FROM dbo.KS_Customers
WHERE region IS NULL;

In [ ]:
UPDATE dbo.KS_Customers
  SET region = N'<None>'
OUTPUT
  deleted.custid,
  deleted.region  AS oldregion,
  inserted.region AS newregion
WHERE region IS NULL;

---
## Proposition 5 - DELETE with OUTPUT
**Problem:** I want to remove all orders placed before August 2020 from my staging orders table. I need to keep a record of what got deleted, so I'll use the OUTPUT clause to return the orderid and orderdate of each deleted row.

In [ ]:
DELETE FROM dbo.KS_Orders
OUTPUT
  deleted.orderid,
  deleted.orderdate
WHERE orderdate < '20200801';

---
## Proposition 6 - DELETE based on JOIN (Brazil customers)
**Problem:** I need to delete all orders in my staging table that were placed by customers from Brazil. Since the country info lives in a different table I will use DELETE with a JOIN to match them up.

In [ ]:
SELECT COUNT(*) AS brazil_orders_before
FROM dbo.KS_Orders AS O
  INNER JOIN Sales.Customers AS C ON O.custid = C.custid
WHERE C.country = N'Brazil';

In [ ]:
DELETE FROM O
FROM dbo.KS_Orders AS O
  INNER JOIN Sales.Customers AS C
    ON O.custid = C.custid
WHERE C.country = N'Brazil';

In [ ]:
SELECT COUNT(*) AS brazil_orders_after
FROM dbo.KS_Orders AS O
  INNER JOIN Sales.Customers AS C ON O.custid = C.custid
WHERE C.country = N'Brazil';

---
## Proposition 7 - UPDATE based on JOIN (UK shipping fix)
**Problem:** For all orders placed by UK customers in my staging table, I want to update the shipcountry, shipregion, and shipcity columns to match the customer's actual country, region, and city from dbo.KS_Customers.

In [ ]:
UPDATE O
SET O.shipcountry = C.country,
    O.shipregion  = C.region,
    O.shipcity    = C.city
FROM dbo.KS_Orders AS O
  INNER JOIN dbo.KS_Customers AS C
    ON O.custid = C.custid
WHERE C.country = N'UK';

In [ ]:
SELECT TOP 10
  O.orderid, O.custid, O.shipcountry, O.shipregion, O.shipcity
FROM dbo.KS_Orders AS O
  INNER JOIN dbo.KS_Customers AS C ON O.custid = C.custid
WHERE C.country = N'UK'
ORDER BY O.orderid;

---
## Proposition 8 - IDENTITY column and INSERT
**Problem:** I want to create a simple audit log table that auto-generates an ID for each row using the IDENTITY property, then insert a few sample log messages and verify the IDs auto-increment.

In [ ]:
DROP TABLE IF EXISTS dbo.KS_AuditLog;
GO
CREATE TABLE dbo.KS_AuditLog
(
  logid   INT          NOT NULL IDENTITY(1,1) PRIMARY KEY,
  msg     NVARCHAR(50) NOT NULL,
  created DATETIME2    NOT NULL DEFAULT(SYSDATETIME())
);
GO
INSERT INTO dbo.KS_AuditLog(msg) VALUES(N'staging tables created');
INSERT INTO dbo.KS_AuditLog(msg) VALUES(N'orders loaded');
INSERT INTO dbo.KS_AuditLog(msg) VALUES(N'brazil orders removed');
SELECT * FROM dbo.KS_AuditLog ORDER BY logid;

---
## Proposition 9 - UPDATE with TOP (small batch)
**Problem:** I want to increase the freight by $5.00 but only for the first 10 orders by orderid. This shows how to do a controlled batch update using a CTE with TOP.

In [ ]:
SELECT TOP 10 orderid, freight
FROM dbo.KS_Orders
ORDER BY orderid;

In [ ]:
WITH batch AS
(
  SELECT TOP(10) freight
  FROM dbo.KS_Orders
  ORDER BY orderid
)
UPDATE batch
  SET freight = freight + 5.00;

In [ ]:
SELECT TOP 10 orderid, freight
FROM dbo.KS_Orders
ORDER BY orderid;

---
## Proposition 10 - TRUNCATE TABLE (with FK handling)
**Problem:** I have a parent table (KS_Parent) and child table (KS_Child) linked by a foreign key. I want to empty both tables efficiently using TRUNCATE instead of DELETE. Since TRUNCATE wont work on a table referenced by a FK I need to drop the constraint first, truncate both, then re-add the FK.

In [ ]:
DROP TABLE IF EXISTS dbo.KS_Child, dbo.KS_Parent;
GO
CREATE TABLE dbo.KS_Parent
(
  pid   INT         NOT NULL PRIMARY KEY,
  pname NVARCHAR(20) NOT NULL
);
CREATE TABLE dbo.KS_Child
(
  cid  INT          NOT NULL PRIMARY KEY,
  pid  INT          NOT NULL,
  cval NVARCHAR(20) NOT NULL,
  CONSTRAINT FK_KS_Child_Parent FOREIGN KEY(pid)
    REFERENCES dbo.KS_Parent(pid)
);
GO
INSERT INTO dbo.KS_Parent VALUES(1, N'Parent A'), (2, N'Parent B');
INSERT INTO dbo.KS_Child  VALUES(10, 1, N'Child X'), (20, 2, N'Child Y'); SELECT 'KS_Parent' AS tbl, COUNT(*) AS row_count FROM dbo.KS_Parent
UNION ALL
SELECT 'KS_Child', COUNT(*) FROM dbo.KS_Child;

In [ ]:
ALTER TABLE dbo.KS_Child DROP CONSTRAINT FK_KS_Child_Parent;
TRUNCATE TABLE dbo.KS_Child;
TRUNCATE TABLE dbo.KS_Parent;
ALTER TABLE dbo.KS_Child ADD CONSTRAINT FK_KS_Child_Parent
  FOREIGN KEY(pid) REFERENCES dbo.KS_Parent(pid);
SELECT 'KS_Parent' AS tbl, COUNT(*) AS row_count FROM dbo.KS_Parent
UNION ALL
SELECT 'KS_Child', COUNT(*) FROM dbo.KS_Child;

---
# Part 2 - Excel PowerQuery Report (5 Queries)

These 5 queries are loaded into Excel using **Data -> Get Data -> From SQL Server Database**.  
Each query goes on its own sheet. After loading, open Power Query to set data types, sort, and transform.  
Then create charts/tables in Excel for the report.

| Sheet | Description |
|-------|-------------|
| Q1 | Total orders and freight grouped by ship country |
| Q2 | Top 20 customers by total revenue |
| Q3 | Monthly revenue trend |
| Q4 | Employee performance (orders + freight) |
| Q5 | Product sales summary (units + revenue) |

### Query 1 - Orders by Ship Country

In [ ]:
SELECT shipcountry,
       COUNT(*)     AS TotalOrders,
       SUM(freight) AS TotalFreight,
       AVG(freight) AS AvgFreight
FROM Sales.Orders
GROUP BY shipcountry
ORDER BY TotalOrders DESC;

### Query 2 - Top 20 Customers by Revenue

In [ ]:
SELECT TOP 20
       C.custid,
       C.companyname,
       C.country,
       COUNT(DISTINCT O.orderid)                        AS OrderCount,
       SUM(OD.qty * OD.unitprice * (1 - OD.discount))  AS TotalRevenue
FROM Sales.Customers AS C
  INNER JOIN Sales.Orders AS O
    ON C.custid = O.custid
  INNER JOIN Sales.OrderDetails AS OD
    ON O.orderid = OD.orderid
GROUP BY C.custid, C.companyname, C.country
ORDER BY TotalRevenue DESC;

### Query 3 - Monthly Revenue Trend

In [ ]:
SELECT YEAR(O.orderdate)                                AS OrderYear,
       MONTH(O.orderdate)                               AS OrderMonth,
       SUM(OD.qty * OD.unitprice * (1 - OD.discount))  AS MonthlyRevenue,
       COUNT(DISTINCT O.orderid)                        AS OrderCount
FROM Sales.Orders AS O
  INNER JOIN Sales.OrderDetails AS OD
    ON O.orderid = OD.orderid
GROUP BY YEAR(O.orderdate), MONTH(O.orderdate)
ORDER BY OrderYear, OrderMonth;

### Query 4 - Employee Performance

In [ ]:
SELECT E.empid,
       E.firstname + ' ' + E.lastname AS EmployeeName,
       E.title,
       COUNT(O.orderid) AS TotalOrders,
       SUM(O.freight)   AS TotalFreight,
       AVG(O.freight)   AS AvgFreight
FROM HR.Employees AS E
  INNER JOIN Sales.Orders AS O
    ON E.empid = O.empid
GROUP BY E.empid, E.firstname, E.lastname, E.title
ORDER BY TotalOrders DESC;

### Query 5 - Product Sales Summary

In [ ]:
SELECT P.productid,
       P.productname,
       P.categoryid,
       SUM(OD.qty)                                      AS UnitsSold,
       SUM(OD.qty * OD.unitprice * (1 - OD.discount))  AS Revenue,
       COUNT(DISTINCT OD.orderid)                       AS TimesOrdered
FROM Production.Products AS P
  INNER JOIN Sales.OrderDetails AS OD
    ON P.productid = OD.productid
GROUP BY P.productid, P.productname, P.categoryid
ORDER BY Revenue DESC;

---
## Cleanup
Drop all the staging tables I created during this assignment.

In [ ]:
DROP TABLE IF EXISTS dbo.KS_Child, dbo.KS_Parent;
DROP TABLE IF EXISTS dbo.KS_AuditLog;
DROP TABLE IF EXISTS dbo.KS_Orders;
DROP TABLE IF EXISTS dbo.KS_Customers;

---
## LLM Footnote
I used an LLM (Claude) to help outline my proposition ideas and double-check my SQL syntax. All queries were written and tested by me against the Northwinds2024Student database in Azure Data Studio.

---
*File: Individual_Group2_HW7_KanwalJitSingh.ipynb*